In [1]:
import duckdb
import pandas as pd

# Run once — sets working directory to project root regardless of where notebook is
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

con = duckdb.connect()
con.execute("CREATE TABLE cms AS SELECT * FROM 'data/processed/cms_clean.csv'")

Working directory: c:\Users\gamer\Documents\projects\cms-medicare-audit-analysis


In [2]:
q1 = con.execute('''
    SELECT
        ROUND(MIN(charge_to_payment_ratio), 2)    AS min_ratio,
        ROUND(AVG(charge_to_payment_ratio), 2)    AS avg_ratio,
        ROUND(MEDIAN(charge_to_payment_ratio), 2) AS median_ratio,
        ROUND(MAX(charge_to_payment_ratio), 2)    AS max_ratio,
        ROUND(STDDEV(charge_to_payment_ratio), 2) AS stddev_ratio
    FROM cms
    WHERE charge_to_payment_ratio IS NOT NULL
''').df()

print(q1)

   min_ratio  avg_ratio  median_ratio  max_ratio  stddev_ratio
0        1.0       6.34          5.52     141.52          3.61


In [3]:
q2 = con.execute('''
    SELECT
        Rndrng_Prvdr_State_Abrvtn                  AS state,
        COUNT(DISTINCT Rndrng_Prvdr_CCN)           AS num_providers,
        ROUND(AVG(charge_to_payment_ratio), 2)     AS avg_ratio,
        ROUND(MEDIAN(charge_to_payment_ratio), 2)  AS median_ratio
    FROM cms
    WHERE charge_to_payment_ratio IS NOT NULL
    GROUP BY Rndrng_Prvdr_State_Abrvtn
    ORDER BY avg_ratio DESC
''').df()

print(q2)
q2.to_csv('outputs/ratio_by_state.csv', index=False)

   state  num_providers  avg_ratio  median_ratio
0     NV             22      11.32         11.05
1     FL            165       9.32          8.13
2     CO             48       8.86          7.89
3     NJ             63       8.73          7.71
4     TX            238       8.11          7.04
5     CA            275       7.83          6.99
6     AZ             57       7.33          6.66
7     AL             68       7.25          5.53
8     KS             45       7.13          6.21
9     AK              8       7.03          6.96
10    PA            129       7.00          6.33
11    VT              6       6.81          5.93
12    SC             51       6.64          5.77
13    GA             88       6.27          5.98
14    TN             77       6.23          5.51
15    OK             67       6.23          5.54
16    LA             76       6.20          5.69
17    DC              6       6.05          5.49
18    KY             60       6.01          5.56
19    VA            

In [4]:
q3 = con.execute('''
    SELECT
        Rndrng_Prvdr_CCN                           AS provider_id,
        Rndrng_Prvdr_Org_Name                      AS hospital_name,
        Rndrng_Prvdr_State_Abrvtn                  AS state,
        DRG_Desc                                   AS drg,
        Tot_Dschrgs                                AS discharges,
        ROUND(Avg_Submtd_Cvrd_Chrg, 2)             AS avg_charge,
        ROUND(Avg_Mdcr_Pymt_Amt, 2)                AS avg_medicare_payment,
        ROUND(charge_to_payment_ratio, 2)          AS ratio
    FROM cms
    WHERE charge_to_payment_ratio IS NOT NULL
    ORDER BY charge_to_payment_ratio DESC
    LIMIT 10
''').df()

print(q3)
q3.to_csv('outputs/top10_highest_ratio.csv', index=False)

  provider_id                             hospital_name state  \
0      470001            CENTRAL VERMONT MEDICAL CENTER    VT   
1      310016          CAREPOINT HEALTH-CHRIST HOSPITAL    NJ   
2      290021            VALLEY HOSPITAL MEDICAL CENTER    NV   
3      310044  CAPITAL HEALTH MEDICAL CENTER - HOPEWELL    NJ   
4      310044  CAPITAL HEALTH MEDICAL CENTER - HOPEWELL    NJ   
5      390164               UPMC PRESBYTERIAN SHADYSIDE    PA   
6      390164               UPMC PRESBYTERIAN SHADYSIDE    PA   
7      190176           EAST JEFFERSON GENERAL HOSPITAL    LA   
8      490118                 HENRICO DOCTORS' HOSPITAL    VA   
9      290022                   DESERT SPRINGS HOSPITAL    NV   

                                                 drg  discharges  avg_charge  \
0  CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...          14    25513.64   
1                             CELLULITIS WITHOUT MCC          11   414963.36   
2   ORGANIC DISTURBANCES AND INTELLECTUAL DI

In [5]:
q4 = con.execute('''
    SELECT
        CASE
            WHEN CAST(DRG_Cd AS INT) BETWEEN 1   AND 103 THEN 'Nervous System'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 113 AND 145 THEN 'Eye & Ear'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 146 AND 159 THEN 'ENT'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 163 AND 208 THEN 'Respiratory'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 215 AND 316 THEN 'Circulatory'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 326 AND 395 THEN 'Digestive'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 405 AND 446 THEN 'Hepatobiliary'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 453 AND 566 THEN 'Musculoskeletal'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 570 AND 585 THEN 'Skin & Tissue'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 614 AND 645 THEN 'Kidney & Urinary'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 650 AND 700 THEN 'Reproductive'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 734 AND 761 THEN 'Alcohol & Drug'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 800 AND 816 THEN 'Infections'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 870 AND 872 THEN 'Septicaemia'
            ELSE 'Other'
        END                                        AS drg_category,
        COUNT(*)                                   AS records,
        ROUND(AVG(charge_to_payment_ratio), 2)     AS avg_ratio,
        ROUND(MEDIAN(charge_to_payment_ratio), 2)  AS median_ratio,
        ROUND(MAX(charge_to_payment_ratio), 2)     AS max_ratio
    FROM cms
    WHERE charge_to_payment_ratio IS NOT NULL
    GROUP BY drg_category
    ORDER BY avg_ratio DESC
''').df()

print(q4)
q4.to_csv('outputs/ratio_by_drg_category.csv', index=False)

        drg_category  records  avg_ratio  median_ratio  max_ratio
0                ENT      429       8.64          7.55      29.72
1     Alcohol & Drug      147       7.91          6.49      38.82
2          Eye & Ear      156       7.19          7.05      21.62
3      Skin & Tissue      136       7.11          6.03      25.93
4     Nervous System    16114       6.76          5.79      42.14
5      Hepatobiliary     4449       6.74          5.96      35.44
6        Circulatory    30649       6.64          5.79     141.52
7          Digestive    15938       6.51          5.70      42.19
8    Musculoskeletal    16870       6.34          5.51      39.53
9         Infections     2821       6.22          5.52      46.93
10  Kidney & Urinary     7244       6.22          5.48      40.15
11      Reproductive    13057       6.13          5.25      53.75
12       Respiratory    18796       5.96          5.23      47.04
13             Other    13648       5.86          5.18      80.36
14       S

In [6]:
q5 = con.execute('''
    WITH volume_tiers AS (
        SELECT
            Rndrng_Prvdr_CCN,
            Rndrng_Prvdr_Org_Name,
            Rndrng_Prvdr_State_Abrvtn,
            Tot_Dschrgs,
            charge_to_payment_ratio,
            NTILE(4) OVER (ORDER BY Tot_Dschrgs) AS volume_quartile
        FROM cms
        WHERE charge_to_payment_ratio IS NOT NULL
    )
    SELECT
        volume_quartile,
        CASE volume_quartile
            WHEN 1 THEN 'Q1 - Lowest Volume'
            WHEN 2 THEN 'Q2 - Low-Mid Volume'
            WHEN 3 THEN 'Q3 - High-Mid Volume'
            WHEN 4 THEN 'Q4 - Highest Volume'
        END                                        AS tier_label,
        COUNT(*)                                   AS records,
        ROUND(AVG(Tot_Dschrgs), 0)                 AS avg_discharges,
        ROUND(AVG(charge_to_payment_ratio), 2)     AS avg_ratio,
        ROUND(MEDIAN(charge_to_payment_ratio), 2)  AS median_ratio,
        ROUND(MAX(charge_to_payment_ratio), 2)     AS max_ratio
    FROM volume_tiers
    GROUP BY volume_quartile
    ORDER BY volume_quartile
''').df()

print(q5)
q5.to_csv('outputs/volume_tier_analysis.csv', index=False)

   volume_quartile            tier_label  records  avg_discharges  avg_ratio  \
0                1    Q1 - Lowest Volume    36563            12.0       6.41   
1                2   Q2 - Low-Mid Volume    36563            17.0       6.36   
2                3  Q3 - High-Mid Volume    36563            26.0       6.38   
3                4   Q4 - Highest Volume    36562            80.0       6.22   

   median_ratio  max_ratio  
0          5.52     141.52  
1          5.50      49.56  
2          5.56      55.30  
3          5.51      40.25  


In [7]:
q6 = con.execute('''
    SELECT
        Rndrng_Prvdr_State_Abrvtn                       AS state,
        COUNT(DISTINCT Rndrng_Prvdr_CCN)                AS total_providers,
        COUNT(*)                                        AS total_drg_records,
        SUM(CAST(ratio_outlier_flag AS INT))            AS outlier_records,
        COUNT(DISTINCT CASE WHEN ratio_outlier_flag = true 
            THEN Rndrng_Prvdr_CCN END)                  AS outlier_providers,
        ROUND(
            SUM(CAST(ratio_outlier_flag AS INT)) * 100.0 / COUNT(*), 1
    )                                                   AS outlier_record_pct
    FROM cms
    GROUP BY Rndrng_Prvdr_State_Abrvtn
    ORDER BY outlier_records DESC;
''').df()

print(q6)
q6.to_csv('outputs/outliers_by_state.csv', index=False)

   state  total_providers  total_drg_records  outlier_records  \
0     FL              165              12009           2118.0   
1     TX              238               9933           1236.0   
2     CA              275              12924           1171.0   
3     NV               22               1305            483.0   
4     NJ               63               4735            466.0   
5     AL               68               2298            300.0   
6     PA              129               6980            269.0   
7     VA               71               4313            262.0   
8     CO               48               1658            228.0   
9     AZ               57               2970            145.0   
10    KS               45               1504            142.0   
11    TN               77               3468            140.0   
12    NY              129               8358            134.0   
13    SC               51               2813            133.0   
14    MO               65

In [8]:
q7 = con.execute('''
    SELECT
        Rndrng_Prvdr_State_Abrvtn                        AS state,
        COUNT(*)                                         AS outlier_records,
        COUNT(DISTINCT Rndrng_Prvdr_CCN)                 AS outlier_providers,
        ROUND(SUM(Avg_Submtd_Cvrd_Chrg * Tot_Dschrgs), 2) AS total_charges,
        ROUND(SUM(Avg_Mdcr_Pymt_Amt * Tot_Dschrgs), 2)    AS total_medicare_paid,
        ROUND(
            SUM(Avg_Submtd_Cvrd_Chrg * Tot_Dschrgs) -
            SUM(Avg_Mdcr_Pymt_Amt * Tot_Dschrgs), 2
        )                                                AS total_gap
    FROM cms
    WHERE ratio_outlier_flag = true
    GROUP BY Rndrng_Prvdr_State_Abrvtn
    ORDER BY total_gap DESC
    LIMIT 15
''').df()

print(q7)
q7.to_csv('outputs/revenue_at_stake_by_state.csv', index=False)

   state  outlier_records  outlier_providers  total_charges  \
0     FL             2118                 95   9.673424e+09   
1     CA             1171                 69   9.476497e+09   
2     TX             1236                 71   8.102852e+09   
3     NV              483                 14   3.206240e+09   
4     NJ              466                 36   2.364680e+09   
5     VA              262                 11   2.032737e+09   
6     PA              269                 31   1.650628e+09   
7     AL              300                 17   1.461816e+09   
8     CO              228                 18   1.448489e+09   
9     KS              142                  7   8.892425e+08   
10    SC              133                 12   6.462564e+08   
11    NY              134                 17   5.573394e+08   
12    AZ              145                 17   4.957466e+08   
13    TN              140                 17   4.957610e+08   
14    MO              110                 10   3.705115

In [9]:
drg_median = con.execute('''
    SELECT
        CASE
            WHEN CAST(DRG_Cd AS INT) BETWEEN 1   AND 103 THEN 'Nervous System'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 113 AND 145 THEN 'Eye & Ear'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 146 AND 159 THEN 'ENT'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 163 AND 208 THEN 'Respiratory'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 215 AND 316 THEN 'Circulatory'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 326 AND 395 THEN 'Digestive'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 405 AND 446 THEN 'Hepatobiliary'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 453 AND 566 THEN 'Musculoskeletal'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 570 AND 585 THEN 'Skin & Tissue'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 614 AND 645 THEN 'Kidney & Urinary'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 650 AND 700 THEN 'Reproductive'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 734 AND 761 THEN 'Alcohol & Drug'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 800 AND 816 THEN 'Infections'
            WHEN CAST(DRG_Cd AS INT) BETWEEN 870 AND 872 THEN 'Septicaemia'
            ELSE 'Other'
        END                                       AS DRG_Category,
        ROUND(MEDIAN(charge_to_payment_ratio), 2) AS median_ratio,
        COUNT(*)                                  AS records
    FROM cms
    GROUP BY DRG_Category
    ORDER BY median_ratio DESC
    LIMIT 5
''').df()

print(drg_median)

     DRG_Category  median_ratio  records
0             ENT          7.55      429
1       Eye & Ear          7.05      156
2  Alcohol & Drug          6.49      147
3   Skin & Tissue          6.03      136
4   Hepatobiliary          5.96     4449


In [10]:
print(con.execute('''
    SELECT
        ROUND(SUM(Avg_Submtd_Cvrd_Chrg * Tot_Dschrgs), 2) AS total_charges,
        ROUND(SUM(Avg_Mdcr_Pymt_Amt * Tot_Dschrgs), 2)    AS total_paid,
        ROUND(SUM((Avg_Submtd_Cvrd_Chrg - Avg_Mdcr_Pymt_Amt) * Tot_Dschrgs), 2) AS total_gap
    FROM cms
    WHERE ratio_outlier_flag = 'True'
''').df())

   total_charges    total_paid     total_gap
0   4.484809e+10  2.783861e+09  4.206423e+10


In [11]:
print(con.execute('''
    SELECT
        ROUND(SUM(Avg_Submtd_Cvrd_Chrg * Tot_Dschrgs), 2) AS total_all_charges,
        ROUND(SUM(Avg_Mdcr_Pymt_Amt * Tot_Dschrgs), 2)    AS total_all_paid
    FROM cms
''').df())

   total_all_charges  total_all_paid
0       4.319502e+11    7.311431e+10
